In [1]:
# ─── Cell 1: Imports & load retriever ────────────────────────────────────────
import json
import pandas as pd
import dill

# load your pages_df so we can sanity-check pages
pages_df = pd.read_pickle("../data/pages_df.pkl")

# load the retriever you saved
with open("../data/tfidf_retriever.dill","rb") as f:
    tfidf_retrieve = dill.load(f)

In [ ]:
# ─── Cell 2: Load test questions ─────────────────────────────────────────────
with open("../data/test_data.json","r",encoding="utf-8") as f:
    test_data = json.load(f)
test_df = pd.DataFrame(test_data)  # expects columns: query, true_page, answer_text

# sanity-check page names
missing = [p for p in test_df.true_page.unique() if p not in pages_df.page_name.values]
assert not missing, "Missing in pages_df: " + ", ".join(missing)

In [ ]:
# ─── Cell 3: Define recall@k ────────────────────────────────────────────────
def recall_at_k(retriever, queries, truths, k=1):
    hits = 0
    for q, true in zip(queries, truths):
        topk = retriever(q, top_k=k)["page_name"].tolist()
        if true in topk:
            hits += 1
    return hits / len(queries)


In [ ]:
# ─── Cell 4: Compute & print ─────────────────────────────────────────────────
for k in (1,3):
    r = recall_at_k(tfidf_retrieve, test_df.query, test_df.true_page, k=k)
    print(f"TF–IDF Recall@{k}: {r:.2%}")

In [ ]:
# ─── Cell 5: (Optional) Error analysis ───────────────────────────────────────
# Show queries that failed at @1
fails = []
for q,true in zip(test_df.query, test_df.true_page):
    top1 = tfidf_retrieve(q, top_k=1)["page_name"].iloc[0]
    if top1 != true:
        fails.append((q,true,top1))
print("\nFailures @1:")
for q,t,got in fails:
    print(f"* Q: {q}\n  exp: {t!r}\n  got: {got!r}\n")